In [87]:
import pandas as pd

In [88]:
df = pd.read_csv('IMDB Dataset.csv')

In [89]:
df.shape

(50000, 2)

In [90]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [91]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [92]:
df.drop_duplicates(inplace = True)

In [93]:
df.shape

(49582, 2)

In [94]:
#pre processing

In [95]:
df['review'] = df['review'].str.lower() #lowercase 

In [96]:
import re

def remove_url(text):
    text = re.sub(r"http\S+", "", text)
    return text

df["review"] = df["review"].apply(remove_url) #removing VRLs
    

In [97]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [98]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [99]:
import nltk 

In [100]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [101]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [102]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [103]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [104]:
from nltk.stem import PorterStemmer

In [105]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [106]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [107]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [108]:
y = df["sentiment"]

In [109]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [110]:
#vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])

In [111]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

In [122]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [123]:
x_train.shape

(39665, 5000)

In [124]:
x_test.shape

(9917, 5000)

In [125]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [126]:
x_train = x_train.toarray()
x_test = x_test.toarray()

In [127]:
train_set = TensorDataset(
    torch.from_numpy(x_train).float(), 
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(x_test).float(), 
    torch.from_numpy(y_test.values).float()
)



In [128]:
train_loader = DataLoader(train_set, shuffle=True, batch_size = 64)
test_loader = DataLoader(test_set, shuffle=True, batch_size = 64)

In [134]:
import torch.nn as nn
import torch.optim as optim

In [135]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
    
        #RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        #fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        #1st val = hidden state of all timestamps
        #2nd val = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out 

In [136]:
input_size = x_train.shape[1]

model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [137]:
#training

In [138]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()
        xb = xb.unsqueeze(1) #add singleton direction
        outputs = model(xb)

        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"{epoch}/{epochs} and loss = {loss.item()}")

0/10 and loss = 0.17888794839382172
1/10 and loss = 0.5057355165481567
2/10 and loss = 0.26073673367500305
3/10 and loss = 0.25477534532546997
4/10 and loss = 0.23091533780097961
5/10 and loss = 0.21995079517364502
6/10 and loss = 0.3360108435153961
7/10 and loss = 0.3071043789386749
8/10 and loss = 0.16241535544395447
9/10 and loss = 0.22505855560302734


In [140]:
#eval

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        outputs = model(xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.63073510134113
